# 5. Stability and molecular packing
**Input:** raw simulation directory and explicit time window.
**Output:** disjoint block summaries, uncertainty sensitivity and species COM RDFs.
**Modules:** trajectory_analysis → stability. No cached reports are required.

These assess selected observables, not universal equilibrium. Temperature regulation alone is insufficient. Potential-energy/pressure logs and independent replicas remain additional checks.

Window selection uses **zero-based source dump frames**: `STARTFRAME` is inclusive and `ENDFRAME` exclusive. The defaults (`0`, `None`) use the entire loaded trajectory. With stride 2, `[3, 9)` selects source frames 4, 6, 8. These indices are not LAMMPS timesteps or rows after striding. Lag and diffusion-fit settings remain in ps.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations_with_replacement
from IPython.display import display
from MartiniFuncs.trajectory_analysis import analyze_run, molecular_window_statistics
from MartiniFuncs.stability import get_series_statistics, get_molecular_com_rdf

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None
DATA_FILE = None
UNITS = "metal"
TIMESTEP_PS = 0.0005
TYPE_SYMBOLS = None
FRAME_STRIDE = 1
MAX_FRAMES = None
TEMPERATURE_DOF = "3N-3"  # unconstrained atoms, 3 COM DOF removed
# Select source frames [STARTFRAME, ENDFRAME), even with FRAME_STRIDE > 1.
STARTFRAME = 0     # inclusive source frame
ENDFRAME = None    # exclusive source frame; None uses the end
N_BLOCKS = 4
BATCH_DURATIONS_PS = [5., 10., 25., 50.]


In [ ]:
result = analyze_run(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE,
                     units=UNITS, timestep_ps=TIMESTEP_PS, type_symbols=TYPE_SYMBOLS,
                     frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES,
                     degrees_of_freedom=TEMPERATURE_DOF, progress=print)
print("Selected files:", result["files"])
window = molecular_window_statistics(result, startframe=STARTFRAME, endframe=ENDFRAME,
                                     max_lag_ps=TIMESTEP_PS)

print("Source frame range [start, end):", window["frame_window"])
frames = pd.DataFrame(result["frames"])
indices = np.flatnonzero(window["frame_mask"])
if N_BLOCKS < 2 or len(indices) < 3*N_BLOCKS:
    raise ValueError("Select at least three frames per block and at least two blocks")
blocks = np.array_split(indices, N_BLOCKS)
dt = np.diff(frames.time_ps.to_numpy()[indices])[0]
batch_sizes = tuple(sorted({max(2,int(round(t/dt))) for t in BATCH_DURATIONS_PS}))
print("Selected window (ps):", window["time_window_ps"])
display(pd.DataFrame([{"block":i,"first_ps":frames.time_ps.iloc[b[0]],"last_ps":frames.time_ps.iloc[b[-1]],
                      "frames":len(b)} for i,b in enumerate(blocks)]))


## Blocks and uncertainty
Standard deviations describe frame fluctuations. Batch-mean standard errors estimate uncertainty only for sufficiently decorrelated batches. Rising errors with larger batches can reveal slow variation; a few large batches give noisy errors. First-zero ACF truncation is heuristic and can miss oscillations or slow tails.

In [ ]:
columns = [k for k in ("temperature_K","density_g_cm3","volume_angstrom3") if k in frames]
rows = []
fig, axes = plt.subplots(len(columns), 2, figsize=(11,3*len(columns)), squeeze=False, layout="constrained")
for row_index,key in enumerate(columns):
    for i,b in enumerate(blocks):
        x = frames[key].to_numpy()[b]
        stats = get_series_statistics(x, sample_interval_ps=dt, batch_sizes=batch_sizes)
        rows.append({"observable":key,"block":i,"mean":stats["mean"],"frame_std":stats["std"]})
        axes[row_index,0].plot(frames.time_ps.iloc[b], x, lw=.7, label=f"block {i}")
    stats = get_series_statistics(frames[key].to_numpy()[indices], sample_interval_ps=dt, batch_sizes=batch_sizes)
    errors = stats["batch_mean_errors"]
    axes[row_index,1].plot([e["batch_duration_ps"] for e in errors],[e["standard_error"] for e in errors],"o-")
    axes[row_index,0].set(xlabel="Time (ps)",ylabel=key)
    axes[row_index,1].set(xlabel="Batch duration (ps)",ylabel="Estimated SE of mean")
display(pd.DataFrame(rows))
plt.show()


## Species COM packing
Use actual wrapped COMs and their instantaneous NPT box, not TOR trajectories. Self pairs are excluded. RDFs are normalized per frame using its volume and available pair count; radii stay below half the shortest box edge. Stable COM packing does not establish stable reactive orientations. Fully periodic orthorhombic cells are required.

In [ ]:
positions = result["wrapped_com_angstrom"]
if positions is None:
    print("RDF unavailable under the current cell convention:", result["notices"])
else:
    boxes = result["box_lengths_angstrom"]
    edges = np.linspace(0, boxes[indices].min()/2*.999, 65)
    nearest_rows = []
    for a,b in combinations_with_replacement(sorted(set(result["species"])),2):
        if a == b and np.sum(result["species"]==a)<2:
            continue
        fig, ax = plt.subplots(figsize=(8,4), layout="constrained")
        for i, block in enumerate(blocks):
            rdf = get_molecular_com_rdf(positions[block],boxes[block],result["species"],
                                        species_a=a,species_b=b,edges=edges)
            ax.plot(rdf["r_angstrom"],rdf["g_r"],label=f"block {i}")
            nearest_rows.append({"central":a,"neighbor":b,"block":i,
                                 "mean_nearest_angstrom":rdf["nearest_distances_angstrom"].mean()})
        ax.set(title=f"{a} / {b}",xlabel="COM separation (angstrom)",ylabel="g(r)")
        ax.legend()
        plt.show()
    display(pd.DataFrame(nearest_rows))
